# Question Answering with LangChain, OpenAI, and MultiQuery Retriever

This interactive workbook demonstrates example of Elasticsearch's [MultiQuery Retriever](https://api.python.langchain.com/en/latest/retrievers/langchain.retrievers.multi_query.MultiQueryRetriever.html) to generate similar queries for a given user input and apply all queries to retrieve a larger set of relevant documents from a vectorstore.

Before we begin, we first split the fictional workplace documents into passages with `langchain` and uses OpenAI to transform these passages into embeddings and then store these into Elasticsearch.

We will then ask a question, generate similar questions using langchain and OpenAI, retrieve relevant passages from the vector store, and use langchain and OpenAI again to provide a summary for the questions.

## Install packages and import modules

In [4]:
pip install langchain-community elasticsearch


   ---------------------------------------- 0.0/929.5 kB ? eta -:--:--
   ---------------------------------------- 929.5/929.5 kB 7.7 MB/s eta 0:00:00

   -------------------- ------------------- 1/2 [elasticsearch]
   -------------------- ------------------- 1/2 [elasticsearch]
   -------------------- ------------------- 1/2 [elasticsearch]
   -------------------- ------------------- 1/2 [elasticsearch]
   -------------------- ------------------- 1/2 [elasticsearch]
   -------------------- ------------------- 1/2 [elasticsearch]
   -------------------- ------------------- 1/2 [elasticsearch]
   -------------------- ------------------- 1/2 [elasticsearch]
   -------------------- ------------------- 1/2 [elasticsearch]
   -------------------- ------------------- 1/2 [elasticsearch]
   -------------------- ------------------- 1/2 [elasticsearch]
   -------------------- ------------------- 1/2 [elasticsearch]
   ---------------------------------------- 2/2 [elasticsearch]

Note: you may n

In [8]:
import sys
print(sys.executable)

c:\Users\Pc\anaconda3\python.exe


In [9]:
!pip install --upgrade pip
!pip install langchain langchain-community langchain-elasticsearch langchain-openai tiktoken jq lark

   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ----------------------------- ---------- 1.3/1.8 MB 7.3 MB/s eta 0:00:01
   ---------------------------------------- 1.8/1.8 MB 6.6 MB/s eta 0:00:00


ERROR: To modify pip, please run the following command:
C:\Users\Pc\anaconda3\python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/926.9 kB ? eta -:--:--
   ---------------------------------------- 926.9/926.9 kB 6.8 MB/s eta 0:00:00

   ---------------------------------------- 0/5 [lark]
   ---------------------------------------- 0/5 [lark]
   ---------------------------------------- 0/5 [lark]
  Attempting uninstall: elastic-transport
   ---------------------------------------- 0/5 [lark]
   ---------------- ----------------------- 2/5 [elastic-transport]
    Found existing installation: elastic-transport 9.1.0
   ---------------- ----------------------- 2/5 [elastic-transport]
    Uninstalling elastic-transport-9.1.0:
   ---------------- ----------------------- 2/5 [elastic-transport]
      Successfully uninstalled elastic-transport-9.1.0
   ---------------- ----------------------- 2/5 [elastic-transport]
   ---------------- ----------------------- 2/5 [elastic-transport]
   ---------------- ----------------------- 2/5 [elastic-transport]
  Attempting uninstall: 

In [13]:
# !python3 -m pip install -qU jq lark langchain langchain-elasticsearch langchain_openai tiktoken

from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_community.vectorstores.elasticsearch import ElasticsearchStore
from langchain_openai.llms import OpenAI
from langchain.retrievers.multi_query import MultiQueryRetriever
from getpass import getpass

## Connect to Elasticsearch

ℹ️ We're using an Elastic Cloud deployment of Elasticsearch for this notebook. If you don't have an Elastic Cloud deployment, sign up [here](https://cloud.elastic.co/registration?utm_source=github&utm_content=elasticsearch-labs-notebook) for a free trial. 

We'll use the **Cloud ID** to identify our deployment, because we are using Elastic Cloud deployment. To find the Cloud ID for your deployment, go to https://cloud.elastic.co/deployments and select your deployment.

We will use [ElasticsearchStore](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html) to connect to our elastic cloud deployment, This would help create and index data easily.  We would also send list of documents that we created in the previous step

In [15]:
# https://www.elastic.co/search-labs/tutorials/install-elasticsearch/elastic-cloud#finding-your-cloud-id
ELASTIC_CLOUD_ID = getpass("Elastic Cloud ID: ")

# https://www.elastic.co/search-labs/tutorials/install-elasticsearch/elastic-cloud#creating-an-api-key
ELASTIC_API_KEY = getpass("Elastic Api Key: ")

# https://platform.openai.com/api-keys
OPENAI_API_KEY = getpass("OpenAI API key: ")

embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)

vectorstore = ElasticsearchStore(
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
    index_name= None, #give it a meaningful name,
    embedding=embeddings,
)

C:\Users\Pc\AppData\Local\Temp\ipykernel_17180\2280905237.py:12: LangChainPendingDeprecationWarning: The class `ElasticsearchStore` will be deprecated in a future version. Use :class:`~Use class in langchain-elasticsearch package` instead.
  vectorstore = ElasticsearchStore(


In [31]:
from urllib.request import urlopen
import json
url = "https://raw.githubusercontent.com/elastic/elasticsearch-labs/main/example-apps/chatbot-rag-app/data/data.json"
response = urlopen(url)
data = json.load(response)
with open("temp.json", "w") as json_file:
    json.dump(data, json_file)

## Indexing Data into Elasticsearch
Let's download the sample dataset and deserialize the document.

In [17]:
from urllib.request import urlopen
import json

url = "https://raw.githubusercontent.com/elastic/elasticsearch-labs/main/example-apps/chatbot-rag-app/data/data.json"

response = urlopen(url)
data = json.load(response)

with open("temp.json", "w") as json_file:
    json.dump(data, json_file)

### Split Documents into Passages

We’ll chunk documents into passages in order to improve the retrieval specificity and to ensure that we can provide multiple passages within the context window of the final question answering prompt.

Here we are chunking documents into 800 token passages with an overlap of 400 tokens.

Here we are using a simple splitter but Langchain offers more advanced splitters to reduce the chance of context being lost.

In [32]:
from dateutil.parser import parse
def metadata_func(record: dict, metadata: dict) -> dict:
    metadata['name'] = record.get('name', 'unknown-source')
    metadata['summary'] = record.get('summary', '')
    metadata['url'] = record.get('url', '')
    metadata['category'] = record.get('category', '')
    updated_at = record.get('updated_at')
    if updated_at and updated_at.strip():
        try:
            dt = parse(updated_at)
            metadata['updated_at'] = dt.isoformat()
        except Exception as e:
            print(f"Warning: Failed to parse date '{updated_at}' - {e}")
            metadata.pop('updated_at', None)
    else:
        metadata.pop('updated_at', None)
    print(f"Processing record, updated_at: {updated_at} -> metadata: {metadata.get('updated_at', 'Not set')}")
    return metadata

In [33]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
loader = JSONLoader(
    file_path="temp.json",
    jq_schema=".[]",
    content_key="content",
    metadata_func=metadata_func,
)
raw_docs = loader.load()
for i, doc in enumerate(raw_docs[:5]):
    print(f"Raw Doc {i+1} content:", doc.page_content)
    print(f"Raw Doc {i+1} metadata:", doc.metadata)
docs = loader.load_and_split(text_splitter=text_splitter)
for i, doc in enumerate(docs[:3]):
    print(f"Doc {i+1} metadata:", doc.metadata)
for doc in docs[:3]:
    print(doc.metadata)  # Should include 'name'
for doc in docs:
    if 'name' not in doc.metadata or not doc.metadata['name']:
        doc.metadata['name'] = "unknown-source"
vectorstore = ElasticsearchStore.from_documents(
    docs,                # these docs with metadata!
    embeddings,
    index_name="search-ipd3",
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
)

Processing record, updated_at: 2020-03-01 -> metadata: 2020-03-01T00:00:00
Processing record, updated_at: 2022-04-29 -> metadata: 2022-04-29T00:00:00
Processing record, updated_at: 2023-05-01 -> metadata: 2023-05-01T00:00:00
Processing record, updated_at: 2023-04-15 -> metadata: 2023-04-15T00:00:00
Processing record, updated_at: 2018-04-16 -> metadata: 2018-04-16T00:00:00
Processing record, updated_at: None -> metadata: Not set
Processing record, updated_at: None -> metadata: Not set
Processing record, updated_at: None -> metadata: Not set
Processing record, updated_at: None -> metadata: Not set
Processing record, updated_at: None -> metadata: Not set
Processing record, updated_at: None -> metadata: Not set
Processing record, updated_at: None -> metadata: Not set
Processing record, updated_at: None -> metadata: Not set
Processing record, updated_at: None -> metadata: Not set
Processing record, updated_at: None -> metadata: Not set
Raw Doc 1 content: Effective: March 2020
Purpose

The p

In [34]:
vectorstore = ElasticsearchStore.from_documents(
    docs,
    embeddings,
    index_name="search-ipd3",
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
)
llm = OpenAI(temperature=0, openai_api_key=OPENAI_API_KEY)
retriever = MultiQueryRetriever.from_llm(vectorstore.as_retriever(), llm)

In [ ]:
# from langchain.document_loaders import JSONLoader
# from langchain.text_splitter import RecursiveCharacterTextSplitter

# # def metadata_func(record: dict, metadata: dict) -> dict:
# #     #Populate the metadata dictionary with keys name, summary, url, category, and updated_at.
# #     metadata["name"] = record.get("name", "")
# #     metadata["summary"] = record.get("summary", "")
# #     metadata["url"] = record.get("url", "")
# #     metadata["category"] = record.get("category", "")
# #     metadata["updated_at"] = record.get("updated_at", "")
# #     return metadata

# from datetime import datetime

# def safe_parse_date(date_str):
#     """Return ISO8601 date or None if invalid."""
#     if not date_str:
#         return None
#     try:
#         return datetime.fromisoformat(date_str).isoformat()
#     except Exception:
#         return None

# def clean_metadata(record: dict, metadata: dict) -> dict:
#     """Clean metadata to avoid Elasticsearch indexing errors."""
#     safe_meta = {}
#     for key in ["name", "summary", "url", "category"]:
#         val = record.get(key)
#         safe_meta[key] = val if val and val != "" else None
    
#     # Handle updated_at field
#     safe_meta["updated_at"] = safe_parse_date(record.get("updated_at"))
#     return safe_meta


# # Load JSON file
# loader = JSONLoader(
#     file_path="temp.json",
#     jq_schema=".[]",
#     content_key="content",
#     metadata_func=metadata_func,
# )

# # Define a text splitter with valid chunk size and overlap
# text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
#     chunk_size=800,   # set a valid number
#     chunk_overlap=400  # set a valid number
# )

# # Load and split documents
# docs = loader.load_and_split(text_splitter=text_splitter)

# docs[0]  # Display the first document to verify

Document(metadata={'source': 'D:\\Ironhack\\Week 7\\DAY 4\\lab-chatbot-with-multi-query-retriever\\temp.json', 'seq_num': 1, 'name': 'Work From Home Policy', 'summary': 'This policy outlines the guidelines for full-time remote work, including eligibility, equipment and resources, workspace requirements, communication expectations, performance expectations, time tracking and overtime, confidentiality and data security, health and well-being, and policy reviews and updates. Employees are encouraged to direct any questions or concerns', 'url': './sharepoint/Work from home policy.txt', 'category': 'teams', 'updated_at': '2020-03-01'}, page_content="Effective: March 2020\nPurpose\n\nThe purpose of this full-time work-from-home policy is to provide guidelines and support for employees to conduct their work remotely, ensuring the continuity and productivity of business operations during the COVID-19 pandemic and beyond.\nScope\n\nThis policy applies to all employees who are eligible for remot

In [ ]:
# from datetime import datetime

# def safe_parse_date(date_str):
#     """Convert to ISO date string or None."""
#     if not date_str:
#         return None
#     try:
#         return datetime.fromisoformat(date_str).isoformat()
#     except Exception:
#         return None

# def clean_metadata(record: dict, metadata: dict) -> dict:
#     safe_meta = {}
#     for key in ["name", "summary", "url", "category"]:
#         value = record.get(key)
#         safe_meta[key] = value if value else None

#     # Handle updated_at separately
#     updated_at = record.get("updated_at")
#     safe_meta["updated_at"] = safe_parse_date(updated_at)
#     return safe_meta


In [29]:
# !python3 -m pip install -qU jq lark langchain langchain-elasticsearch langchain_openai tiktoken

from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_community.vectorstores.elasticsearch import ElasticsearchStore
from langchain_openai.llms import OpenAI
from langchain.retrievers.multi_query import MultiQueryRetriever
from getpass import getpass

### Bulk Import Passages

Now that we have split each document into the chunk size of 800, we will now index data to elasticsearch using [ElasticsearchStore.from_documents](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html#langchain.vectorstores.elasticsearch.ElasticsearchStore.from_documents).

We will use Cloud ID, Password and Index name values set in the `Create cloud deployment` step.

In [35]:
documents = vectorstore.from_documents(
    docs,
    embeddings,
    index_name= "elastic_index",
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
)

llm = OpenAI(temperature=0, openai_api_key=OPENAI_API_KEY)

retriever = MultiQueryRetriever.from_llm(vectorstore.as_retriever(), llm)

# Question Answering with MultiQuery Retriever

Now that we have the passages stored in Elasticsearch, we can now ask a question to get the relevant passages.

In [ ]:
from langchain.schema.runnable import RunnableParallel, RunnablePassthrough
from langchain.prompts import ChatPromptTemplate, PromptTemplate
from langchain.schema import format_document

import logging

logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

LLM_CONTEXT_PROMPT = ChatPromptTemplate.from_template(
    """You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Be as verbose and educational in your response as possible. 
    
    context: {context}
    Question: "{question}"  
    Answer:
    """
)

LLM_DOCUMENT_PROMPT = PromptTemplate.from_template(
    """
---
SOURCE: {name}
{page_content}
---
"""
)


def _combine_documents(
    docs, document_prompt=LLM_DOCUMENT_PROMPT, document_separator="\n\n"
):
    doc_strings = [format_document(doc, document_prompt) for doc in docs]
    return document_separator.join(doc_strings)


_context = RunnableParallel(
    context=retriever | _combine_documents,
    question=RunnablePassthrough(),
)

chain = _context | LLM_CONTEXT_PROMPT | llm

ans = chain.invoke("what is the nasa sales team?")

print("---- Answer ----")
print(ans)

INFO:langchain.retrievers.multi_query:Generated queries: ['1. Can you provide information on the sales team at NASA?', '2. How does the sales team at NASA operate?', '3. What are the responsibilities of the sales team at NASA?']


---- Answer ----
The NASA sales team is a part of the Americas region in the sales organization. It is led by two Area Vice-Presidents, Laura Martinez and Gary Johnson, who are responsible for managing dedicated account managers, sales representatives, and support staff. The team's main objectives are to identify and pursue new business opportunities, maintain existing client relationships, and ensure customer satisfaction. They work closely with other departments, such as marketing, product development, and customer support, to deliver high-quality products and services to clients. As an engineer, it is important to understand the goals and objectives of the sales team in order to provide them with the necessary support and information to successfully sell the company's products and services. Effective communication and collaboration with the sales team is crucial in achieving this goal.


**Generate at least two new iteratioins of the previous cells - Be creative.** Did you master Multi-
Query Retriever concepts through this lab?

In [40]:
_context_with_conf = RunnableParallel(
    context=retriever | _combine_documents,
    question=RunnablePassthrough(),
)
chain_with_conf = _context_with_conf | LLM_CONTEXT_PROMPT | llm

ans_conf = chain_with_conf.invoke("Summarize NASA's team structure.")

print("---- Answer ----")
print(ans_conf)


INFO:langchain.retrievers.multi_query:Generated queries: ["1. Can you provide an overview of the organizational structure of NASA's team?", "2. How is NASA's team organized and structured?", '3. What is the team structure of NASA and how does it function?']


---- Answer ----

NASA's team structure consists of two Area Vice-Presidents, Laura Martinez for North America and Gary Johnson for South America. They lead dedicated account managers, sales representatives, and support staff in their respective regions. The team's responsibilities include identifying and pursuing new business opportunities, nurturing existing client relationships, and ensuring customer satisfaction. They also collaborate closely with other departments, such as marketing, product development, and customer support, to consistently deliver high-quality products and services to clients.


In [41]:
from langchain.prompts import ChatPromptTemplate

LLM_REASONING_PROMPT = ChatPromptTemplate.from_template(
    """You are an expert researcher. Think step-by-step:
1. Restate the question clearly.
2. Analyze the context carefully.
3. Provide your reasoning.
4. Give a final, comprehensive answer.

Context: {context}
Question: "{question}"
Answer (with reasoning):
"""
)


In [43]:
chain_reasoning = _context | LLM_REASONING_PROMPT | llm

ans_reasoning = chain_reasoning.invoke("Who are NASA's key decision-makers?")

print("---- Answer ----")
print(ans_reasoning)



INFO:langchain.retrievers.multi_query:Generated queries: ['1. Who holds the most influence in decision-making at NASA?', '2. What individuals are responsible for making key decisions at NASA?', '3. Can you provide a list of the main decision-makers at NASA?']


---- Answer ----

The key decision-makers for NASA (North America South America) in the sales organization are the two Area Vice-Presidents, Laura Martinez and Gary Johnson. This can be inferred from the context provided, which states that the NASA region has two Area Vice-Presidents, and they are responsible for leading their respective regional sales teams. Additionally, the Area Vice-Presidents are responsible for identifying and pursuing new business opportunities, nurturing existing client relationships, and ensuring customer satisfaction, which are all key decision-making responsibilities. Therefore, it can be concluded that Laura Martinez and Gary Johnson are the key decision-makers for NASA in the sales organization.
